# MyDigitalTwin — Instagram
**Notebook 03 — Ingestion, exploration, nettoyage → Parquet**

Sources :
- `your_instagram_activity/saved/saved_posts.json` → posts sauvegardés
- `your_instagram_activity/comments/post_comments_*.json` → commentaires publics
- `your_instagram_activity/likes/liked_posts.json` → posts likés
- `your_instagram_activity/messages/inbox/*/message_*.json` → métadonnées messages

Outputs :
- `data/parquet/instagram_saved.parquet`
- `data/parquet/instagram_comments.parquet`
- `data/parquet/instagram_likes.parquet`
- `data/parquet/instagram_messages_meta.parquet`

## Objectifs ML
- **Clone NLP (axe 1)** : corpus de tes commentaires pour TF-IDF / N-grams
- **ALS (axe 2)** : signal d'intérêt via les likes
- **K-Means (axe 3)** : activité temporelle (heure, jour)

## 0. Initialisation Spark

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname('__file__'), '../../..')))
from config import RAW_DATA, WAREHOUSE

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import os
import json
import glob

spark = SparkSession.builder \
    .appName("MyDigitalTwin - Instagram") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog",
    ) \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version : {spark.version}")

# Chemins
IG_ROOT     = os.path.join(RAW_DATA, "INSTAGRAM", "your_instagram_activity")
PARQUET_DIR = os.path.join(os.path.dirname(os.path.abspath("__file__")), "..", "..", "data", "parquet")

Spark version : 3.5.5


26/04/19 00:01:58 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


---
## PARTIE 1 — Commentaires
### 1.1 Ingestion

In [2]:
# Collecter tous les fichiers post_comments_*.json
comments_files = glob.glob(f"{IG_ROOT}/comments/post_comments_*.json")
print(f"Fichiers commentaires trouvés : {len(comments_files)}")

# Lire et aplatir en Python d'abord (structure trop imbriquée pour spark.read.json)
def parse_comments(files):
    rows = []
    for path in files:
        with open(path, encoding="utf-8", errors="replace") as f:
            data = json.load(f)
        for item in data:
            smd = item.get("string_map_data", {})
            comment = smd.get("Comment", {}).get("value", "")
            owner   = smd.get("Media Owner", {}).get("value", "")
            ts      = smd.get("Time", {}).get("timestamp", 0)
            if comment:
                rows.append({
                    "text":       comment,
                    "media_owner": owner,
                    "timestamp":  ts,
                })
    return rows

comments_rows = parse_comments(comments_files)
print(f"Commentaires extraits : {len(comments_rows):,}")
if comments_rows:
    print("Exemple :", comments_rows[0])

Fichiers commentaires trouvés : 1
Commentaires extraits : 28
Exemple : {'text': 'DJ', 'media_owner': 'rijmusic_', 'timestamp': 1772835273}


In [3]:
# Créer le DataFrame PySpark
schema_comments = StructType([
    StructField("text",        StringType(), True),
    StructField("media_owner", StringType(), True),
    StructField("timestamp",   LongType(),   True),
])

df_comments = spark.createDataFrame(comments_rows, schema=schema_comments)

# Champs temporels
df_comments = df_comments \
    .withColumn("event_date",    F.to_timestamp(F.col("timestamp"))) \
    .withColumn("event_year",    F.year("event_date")) \
    .withColumn("event_month",   F.date_format("event_date", "yyyy-MM")) \
    .withColumn("event_hour",    F.hour("event_date")) \
    .withColumn("event_weekday", F.dayofweek("event_date")) \
    .withColumn("char_count",    F.length("text")) \
    .withColumn("word_count",    F.size(F.split(F.trim("text"), r"\s+"))) \
    .withColumn("emoji_count",   F.size(F.array_remove(
        F.split(F.regexp_replace("text", r"[\w\s.,!?;:'\"\-()]", " "), " "),
        ""
    ))) \
    .withColumn("platform",      F.lit("instagram")) \
    .withColumn("content_type",  F.lit("comment"))

print(f"Lignes : {df_comments.count():,}")
df_comments.printSchema()
df_comments.show(5, truncate=60)

Lignes : 28
root
 |-- text: string (nullable = true)
 |-- media_owner: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- event_date: timestamp (nullable = true)
 |-- event_year: integer (nullable = true)
 |-- event_month: string (nullable = true)
 |-- event_hour: integer (nullable = true)
 |-- event_weekday: integer (nullable = true)
 |-- char_count: integer (nullable = true)
 |-- word_count: integer (nullable = false)
 |-- emoji_count: integer (nullable = false)
 |-- platform: string (nullable = false)
 |-- content_type: string (nullable = false)

+--------+-----------+----------+-------------------+----------+-----------+----------+-------------+----------+----------+-----------+---------+------------+
|    text|media_owner| timestamp|         event_date|event_year|event_month|event_hour|event_weekday|char_count|word_count|emoji_count| platform|content_type|
+--------+-----------+----------+-------------------+----------+-----------+----------+-------------+-------

### 1.2 Exploration

In [4]:
print("=== Statistiques texte ===")
df_comments.agg(
    F.avg("char_count").alias("avg_chars"),
    F.avg("word_count").alias("avg_words"),
    F.max("char_count").alias("max_chars"),
).show()

print("\n=== Commentaires par année ===")
df_comments.groupBy("event_year").count().orderBy("event_year").show()

print("\n=== Comptes les plus commentés ===")
df_comments.groupBy("media_owner") \
    .count() \
    .orderBy(F.desc("count")) \
    .limit(10) \
    .show()

print("\n=== Activité par heure ===")
df_comments.groupBy("event_hour").count().orderBy("event_hour").show()

=== Statistiques texte ===
+---------+------------------+---------+
|avg_chars|         avg_words|max_chars|
+---------+------------------+---------+
|    14.75|2.2142857142857144|       54|
+---------+------------------+---------+


=== Commentaires par année ===


+----------+-----+
|event_year|count|
+----------+-----+
|      2025|   18|
|      2026|   10|
+----------+-----+


=== Comptes les plus commentés ===
+----------------+-----+
|     media_owner|count|
+----------------+-----+
|     madaclub.be|   18|
|       mlk__frmn|    2|
|        3li0tttt|    2|
|       rijmusic_|    1|
|  kromatic.music|    1|
|    danstamaison|    1|
|booktoneventafro|    1|
|    lexar_global|    1|
|     loumont.xyz|    1|
+----------------+-----+


=== Activité par heure ===
+----------+-----+
|event_hour|count|
+----------+-----+
|         1|    1|
|         7|    1|
|        10|    1|
|        12|    1|
|        14|    3|
|        15|    1|
|        16|    2|
|        17|    4|
|        18|    3|
|        19|    6|
|        21|    4|
|        22|    1|
+----------+-----+



In [5]:
print("=== Top 20 mots les plus utilisés ===")
df_comments \
    .withColumn("word", F.explode(F.split(F.lower(F.col("text")), r"\s+"))) \
    .filter(F.length("word") > 2) \
    .filter(~F.col("word").rlike(r"[^a-záàâäéèêëîïôùûüç']")) \
    .groupBy("word") \
    .count() \
    .orderBy(F.desc("count")) \
    .limit(20) \
    .show()

=== Top 20 mots les plus utilisés ===
+------------+-----+
|        word|count|
+------------+-----+
|       c'est|    2|
|inadmissible|    1|
|         par|    1|
|      contre|    1|
|         nan|    1|
|     canette|    1|
|         ami|    1|
|         mon|    1|
|jagermeister|    1|
|       leave|    1|
|       alone|    1|
|         lit|    1|
|        fait|    1|
|         ton|    1|
|       let's|    1|
|        bien|    1|
|        pour|    1|
|     l'hiver|    1|
|         pas|    1|
|      let'ss|    1|
+------------+-----+



### 1.3 Écriture Parquet

In [6]:
df_comments.write.format("delta").mode("overwrite").save(os.path.join(WAREHOUSE, "instagram_comments"))
print(f"instagram_comments -- {df_comments.count():,} lignes")

26/04/19 00:02:11 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


instagram_comments -- 28 lignes


---
## PARTIE 2 — Likes
### 2.1 Ingestion

In [7]:
likes_path = f"{IG_ROOT}/likes/liked_posts.json"

def parse_likes(path):
    rows = []
    with open(path, encoding="utf-8", errors="replace") as f:
        data = json.load(f)
    for item in data:
        ts  = item.get("timestamp", 0)
        url = ""
        # Extraire l'URL du post liké
        for lv in item.get("label_values", []):
            if lv.get("label") == "URL":
                url = lv.get("value", lv.get("href", ""))
                break
        rows.append({"timestamp": ts, "post_url": url})
    return rows

likes_rows = parse_likes(likes_path)
print(f"Likes extraits : {len(likes_rows):,}")
print("Exemple :", likes_rows[0])

Likes extraits : 17,535
Exemple : {'timestamp': 1773991016, 'post_url': 'https://www.instagram.com/p/DWF5G1nDEWE/'}


In [8]:
schema_likes = StructType([
    StructField("timestamp", LongType(),  True),
    StructField("post_url",  StringType(), True),
])

df_likes = spark.createDataFrame(likes_rows, schema=schema_likes)

df_likes = df_likes \
    .withColumn("event_date",    F.to_timestamp(F.col("timestamp"))) \
    .withColumn("event_year",    F.year("event_date")) \
    .withColumn("event_month",   F.date_format("event_date", "yyyy-MM")) \
    .withColumn("event_hour",    F.hour("event_date")) \
    .withColumn("event_weekday", F.dayofweek("event_date")) \
    .withColumn("platform",      F.lit("instagram")) \
    .withColumn("action_type",   F.lit("like")) \
    .withColumn("interaction_weight", F.lit(2.0))

print(f"Lignes : {df_likes.count():,}")
df_likes.show(5, truncate=60)

Lignes : 17,535
+----------+-------------------------------------------+-------------------+----------+-----------+----------+-------------+---------+-----------+------------------+
| timestamp|                                   post_url|         event_date|event_year|event_month|event_hour|event_weekday| platform|action_type|interaction_weight|
+----------+-------------------------------------------+-------------------+----------+-----------+----------+-------------+---------+-----------+------------------+
|1773991016|   https://www.instagram.com/p/DWF5G1nDEWE/|2026-03-20 07:16:56|      2026|    2026-03|         7|            6|instagram|       like|               2.0|
|1773990929|https://www.instagram.com/reel/DWF716iDIj5/|2026-03-20 07:15:29|      2026|    2026-03|         7|            6|instagram|       like|               2.0|
|1773957091|   https://www.instagram.com/p/DWEFNZ0Ao4w/|2026-03-19 21:51:31|      2026|    2026-03|        21|            5|instagram|       like|        

### 2.2 Exploration

In [9]:
print("=== Likes par année ===")
df_likes.groupBy("event_year").count().orderBy("event_year").show()

print("\n=== Likes par heure ===")
df_likes.groupBy("event_hour").count().orderBy("event_hour").show()

print("\n=== Likes par jour de la semaine ===")
df_likes.groupBy("event_weekday").count().orderBy("event_weekday").show()

print("\n=== Mois les plus actifs ===")
df_likes.groupBy("event_month") \
    .count() \
    .orderBy(F.desc("count")) \
    .limit(10) \
    .show()

=== Likes par année ===
+----------+-----+
|event_year|count|
+----------+-----+
|      2023| 2770|
|      2024| 4807|
|      2025| 8090|
|      2026| 1868|
+----------+-----+


=== Likes par heure ===
+----------+-----+
|event_hour|count|
+----------+-----+
|         0|  508|
|         1|  352|
|         2|  196|
|         3|   85|
|         4|  170|
|         5|  243|
|         6|  378|
|         7|  474|
|         8|  591|
|         9|  767|
|        10|  885|
|        11|  904|
|        12|  804|
|        13|  797|
|        14|  707|
|        15|  817|
|        16| 1057|
|        17|  977|
|        18| 1170|
|        19| 1193|
+----------+-----+
only showing top 20 rows


=== Likes par jour de la semaine ===
+-------------+-----+
|event_weekday|count|
+-------------+-----+
|            1| 2570|
|            2| 2592|
|            3| 2608|
|            4| 2530|
|            5| 2537|
|            6| 2421|
|            7| 2277|
+-------------+-----+


=== Mois les plus actifs ===
+----

### 2.3 Écriture Parquet

In [10]:
df_likes.write.format("delta").mode("overwrite").save(os.path.join(WAREHOUSE, "instagram_likes"))
print(f"instagram_likes -- {df_likes.count():,} lignes")

instagram_likes -- 17,535 lignes


---
## PARTIE 3 — Messages (métadonnées uniquement)

> **Confidentialité** : on ne stocke PAS le contenu des messages.  
> On extrait uniquement : timestamp, longueur, type de média, conversation anonymisée.

### 3.1 Ingestion

In [11]:
import hashlib

def anonymize(val, salt="mydigitaltwin"):
    return hashlib.sha256(f"{salt}{val}".encode()).hexdigest()[:10]

inbox_root = f"{IG_ROOT}/messages/inbox"
msg_files  = glob.glob(f"{inbox_root}/*/message_*.json")
print(f"Fichiers messages trouvés : {len(msg_files):,}")

def parse_messages(files, my_name="arnvudl"):
    rows_meta = []
    rows_text = []

    for path in files:
        conv_folder = os.path.basename(os.path.dirname(path))
        conv_id     = anonymize(conv_folder)

        with open(path, encoding="utf-8", errors="replace") as f:
            data = json.load(f)

        is_group     = len(data.get("participants", [])) > 2
        participants = len(data.get("participants", []))

        for msg in data.get("messages", []):
            sender_name = msg.get("sender_name", "")
            ts          = msg.get("timestamp_ms", 0)
            content     = msg.get("content", "")
            is_unsent   = msg.get("is_unsent", False)

            # Type de message
            if is_unsent:
                msg_type = "unsent"
            elif msg.get("photos"):
                msg_type = "photo"
            elif msg.get("videos"):
                msg_type = "video"
            elif msg.get("audio_files"):
                msg_type = "audio"
            elif msg.get("share"):
                msg_type = "share"
            elif content:
                msg_type = "text"
            else:
                msg_type = "other"

            # Métadonnées (tout le monde)
            rows_meta.append({
                "conv_id":      conv_id,
                "is_group":     is_group,
                "participants": participants,
                "sender_anon":  anonymize(sender_name),
                "timestamp_ms": ts,
                "msg_type":     msg_type,
                "char_count":   len(content) if content else 0,
            })

            # Texte — uniquement tes messages à toi
            if content and not is_unsent and sender_name == my_name:
                rows_text.append({
                    "text":       content,
                    "timestamp":  ts,
                    "is_group":   is_group,
                    "platform":   "instagram",
                    "content_type": "dm",
                })

    return rows_meta, rows_text


print("Parsing en cours...")
msg_rows, text_rows = parse_messages(msg_files, my_name="arnvudl")
print(f"Messages (métadonnées) : {len(msg_rows):,}")
print(f"Tes messages (texte)   : {len(text_rows):,}")

Fichiers messages trouvés : 424
Parsing en cours...
Messages (métadonnées) : 368,542
Tes messages (texte)   : 0


In [12]:
schema_msgs = StructType([
    StructField("conv_id",      StringType(),  True),
    StructField("is_group",     BooleanType(), True),
    StructField("participants", IntegerType(), True),
    StructField("sender_anon",  StringType(),  True),
    StructField("timestamp_ms", LongType(),    True),
    StructField("msg_type",     StringType(),  True),
    StructField("char_count",   IntegerType(), True),
])

df_msgs = spark.createDataFrame(msg_rows, schema=schema_msgs)

# Timestamp en ms → secondes pour PySpark
df_msgs = df_msgs \
    .withColumn("event_date",    F.to_timestamp(F.col("timestamp_ms") / 1000)) \
    .withColumn("event_year",    F.year("event_date")) \
    .withColumn("event_month",   F.date_format("event_date", "yyyy-MM")) \
    .withColumn("event_hour",    F.hour("event_date")) \
    .withColumn("event_weekday", F.dayofweek("event_date")) \
    .withColumn("platform",      F.lit("instagram"))

print(f"Lignes : {df_msgs.count():,}")
df_msgs.printSchema()
df_msgs.show(5)

Lignes : 368,542
root
 |-- conv_id: string (nullable = true)
 |-- is_group: boolean (nullable = true)
 |-- participants: integer (nullable = true)
 |-- sender_anon: string (nullable = true)
 |-- timestamp_ms: long (nullable = true)
 |-- msg_type: string (nullable = true)
 |-- char_count: integer (nullable = true)
 |-- event_date: timestamp (nullable = true)
 |-- event_year: integer (nullable = true)
 |-- event_month: string (nullable = true)
 |-- event_hour: integer (nullable = true)
 |-- event_weekday: integer (nullable = true)
 |-- platform: string (nullable = false)

+----------+--------+------------+-----------+-------------+--------+----------+--------------------+----------+-----------+----------+-------------+---------+
|   conv_id|is_group|participants|sender_anon| timestamp_ms|msg_type|char_count|          event_date|event_year|event_month|event_hour|event_weekday| platform|
+----------+--------+------------+-----------+-------------+--------+----------+--------------------+--

### 3.2 Exploration

In [13]:
print("=== Répartition par type de message ===")
df_msgs.groupBy("msg_type").count().orderBy(F.desc("count")).show()

print("\n=== Groupes vs conversations privées ===")
df_msgs.groupBy("is_group").count().show()

print("\n=== Messages par année ===")
df_msgs.groupBy("event_year").count().orderBy("event_year").show()

print("\n=== Activité par heure ===")
df_msgs.groupBy("event_hour").count().orderBy("event_hour").show()

print("\n=== Top 10 conversations les plus actives ===")
df_msgs.groupBy("conv_id", "is_group", "participants") \
    .count() \
    .orderBy(F.desc("count")) \
    .limit(10) \
    .show()

print("\n=== Longueur moyenne des messages texte ===")
df_msgs.filter(F.col("msg_type") == "text") \
    .agg(
        F.avg("char_count").alias("avg_chars"),
        F.max("char_count").alias("max_chars"),
    ).show()

=== Répartition par type de message ===
+--------+------+
|msg_type| count|
+--------+------+
|    text|342641|
|   audio|  6764|
|   other|  6435|
|   share|  6013|
|   photo|  5521|
|   video|  1161|
|  unsent|     7|
+--------+------+


=== Groupes vs conversations privées ===
+--------+------+
|is_group| count|
+--------+------+
|    true|100866|
|   false|267676|
+--------+------+


=== Messages par année ===
+----------+------+
|event_year| count|
+----------+------+
|      2023| 64840|
|      2024|125088|
|      2025|166469|
|      2026| 12145|
+----------+------+


=== Activité par heure ===
+----------+-----+
|event_hour|count|
+----------+-----+
|         0|10524|
|         1| 4128|
|         2| 1240|
|         3|  434|
|         4|  539|
|         5| 1651|
|         6| 2790|
|         7| 5241|
|         8| 7800|
|         9|11664|
|        10|13835|
|        11|18590|
|        12|18229|
|        13|18360|
|        14|18050|
|        15|20779|
|        16|24009|
|        17|2

### 3.3 Écriture Parquet

In [14]:
df_msgs.write.format("delta").mode("overwrite").save(os.path.join(WAREHOUSE, "instagram_messages_meta"))
print(f"instagram_messages_meta -- {df_msgs.count():,} lignes")

instagram_messages_meta -- 368,542 lignes


---
## PARTIE 4 — Saved Posts

> Posts sauvegardés = signal d'intérêt **fort** (weight=3 pour ALS).  
> On conserve le `href` pour pouvoir retrouver le post original.

### 4.1 Ingestion

In [15]:
saved_path = f"{IG_ROOT}/saved/saved_posts.json"

def parse_saved(path):
    rows = []
    with open(path, encoding="utf-8", errors="replace") as f:
        data = json.load(f)
    for item in data.get("saved_saved_media", []):
        title = item.get("title", "")
        saved_on = item.get("string_map_data", {}).get("Saved on", {})
        href  = saved_on.get("href", "")
        ts    = saved_on.get("timestamp", 0)
        rows.append({
            "account":   title,
            "post_href": href,
            "timestamp": ts,
        })
    return rows

saved_rows = parse_saved(saved_path)
print(f"Posts sauvegardés : {len(saved_rows):,}")
print("Exemple :", saved_rows[0])

Posts sauvegardés : 13
Exemple : {'account': 'ibizastardustradio', 'post_href': 'https://www.instagram.com/p/DVf98jVDGp3/', 'timestamp': 1773247684}


In [16]:
schema_saved = StructType([
    StructField("account",   StringType(), True),
    StructField("post_href", StringType(), True),
    StructField("timestamp", LongType(),   True),
])

df_saved = spark.createDataFrame(saved_rows, schema=schema_saved)

df_saved = df_saved \
    .withColumn("event_date",    F.to_timestamp(F.col("timestamp"))) \
    .withColumn("event_year",    F.year("event_date")) \
    .withColumn("event_month",   F.date_format("event_date", "yyyy-MM")) \
    .withColumn("event_hour",    F.hour("event_date")) \
    .withColumn("event_weekday", F.dayofweek("event_date")) \
    .withColumn("platform",      F.lit("instagram")) \
    .withColumn("action_type",   F.lit("saved")) \
    .withColumn("interaction_weight", F.lit(3.0))

print(f"Lignes : {df_saved.count():,}")
df_saved.show(5, truncate=70)

Lignes : 13
+--------------------+-------------------------------------------+----------+-------------------+----------+-----------+----------+-------------+---------+-----------+------------------+
|             account|                                  post_href| timestamp|         event_date|event_year|event_month|event_hour|event_weekday| platform|action_type|interaction_weight|
+--------------------+-------------------------------------------+----------+-------------------+----------+-----------+----------+-------------+---------+-----------+------------------+
|  ibizastardustradio|   https://www.instagram.com/p/DVf98jVDGp3/|1773247684|2026-03-11 16:48:04|      2026|    2026-03|        16|            4|instagram|      saved|               3.0|
|thehybriddesigner.np|   https://www.instagram.com/p/DVJAZ3niFq-/|1773239579|2026-03-11 14:32:59|      2026|    2026-03|        14|            4|instagram|      saved|               3.0|
|      brillyondabeat|   https://www.instagram.com/p/

### 4.2 Exploration

In [17]:
print("=== Posts sauvegardés par année ===")
df_saved.groupBy("event_year").count().orderBy("event_year").show()

print("\n=== Top 15 comptes dont tu sauvegardes le plus ===")
df_saved.groupBy("account") \
    .count() \
    .orderBy(F.desc("count")) \
    .limit(15) \
    .show(truncate=40)

print("\n=== Activité par heure ===")
df_saved.groupBy("event_hour").count().orderBy("event_hour").show()

print("\n=== Mois les plus actifs ===")
df_saved.groupBy("event_month") \
    .count() \
    .orderBy(F.desc("count")) \
    .limit(10) \
    .show()

=== Posts sauvegardés par année ===
+----------+-----+
|event_year|count|
+----------+-----+
|      2024|    3|
|      2025|    2|
|      2026|    8|
+----------+-----+


=== Top 15 comptes dont tu sauvegardes le plus ===
+--------------------+-----+
|             account|count|
+--------------------+-----+
|         viewsfrance|    2|
|  ibizastardustradio|    1|
|thehybriddesigner.np|    1|
|      brillyondabeat|    1|
|    kellybadakdesign|    1|
|         djmc_gaz974|    1|
|          alyxxcould|    1|
|   livingthedream.wa|    1|
|           fitwcurly|    1|
|       madameb0nplan|    1|
|          film.booth|    1|
|           woahpaolo|    1|
+--------------------+-----+


=== Activité par heure ===
+----------+-----+
|event_hour|count|
+----------+-----+
|         7|    1|
|        11|    1|
|        14|    2|
|        15|    1|
|        16|    4|
|        17|    1|
|        20|    1|
|        22|    1|
|        23|    1|
+----------+-----+


=== Mois les plus actifs ===
+------

### 4.3 Écriture Parquet

In [18]:
df_saved.write.format("delta").mode("overwrite").save(os.path.join(WAREHOUSE, "instagram_saved"))
print(f"instagram_saved -- {df_saved.count():,} lignes")

instagram_saved -- 13 lignes


In [19]:

# ── PARTIE 5 — Posts vus (algo Meta) ──────────────────────────────────────────
# posts_viewed.json → signal de consommation passif (weight=0.5)
import glob as _glob

def _find_ig_file(patterns):
    """Trouve un fichier Instagram par liste de patterns glob."""
    for p in patterns:
        matches = _glob.glob(p)
        if matches:
            return matches[0]
    return None

posts_viewed_path = _find_ig_file([
    f"{IG_ROOT}/ads_and_topics/posts_viewed.json",
    f"{IG_ROOT}/impressions/posts_viewed.json",
    f"{IG_ROOT}/your_topics/posts_viewed.json",
    os.path.join(os.path.dirname(IG_ROOT), "ads_information", "ads_and_topics", "posts_viewed.json"),
])

def parse_posts_viewed(path):
    rows = []
    if not path:
        return rows
    with open(path, encoding="utf-8", errors="replace") as f:
        data = json.load(f)
    items = data if isinstance(data, list) else (data.get("impressions_history_posts_seen") or [])
    for item in items:
        ts = item.get("timestamp", 0)
        if ts:
            rows.append({"author": "", "timestamp": int(ts)})
    return rows

posts_rows = parse_posts_viewed(posts_viewed_path)
print(f"Posts vus : {len(posts_rows):,}")

schema_pv = StructType([
    StructField("author",    StringType(), True),
    StructField("timestamp", LongType(),   True),
])

if posts_rows:
    df_posts_viewed = spark.createDataFrame(posts_rows, schema=schema_pv) \
        .withColumn("event_date",    F.to_timestamp(F.col("timestamp"))) \
        .withColumn("event_year",    F.year("event_date")) \
        .withColumn("event_month",   F.date_format("event_date", "yyyy-MM")) \
        .withColumn("event_hour",    F.hour("event_date")) \
        .withColumn("event_weekday", F.dayofweek("event_date")) \
        .withColumn("platform",      F.lit("instagram")) \
        .withColumn("action_type",   F.lit("post_viewed")) \
        .withColumn("interaction_weight", F.lit(0.5))

    df_posts_viewed.write.format("delta").mode("overwrite").save(os.path.join(WAREHOUSE, "instagram_posts_viewed"))
    print(f"instagram_posts_viewed -- {df_posts_viewed.count():,} lignes")
else:
    print("⚠ posts_viewed.json introuvable ou vide — fichier skippé")

# ── PARTIE 6 — Vidéos regardées ───────────────────────────────────────────────
videos_watched_path = _find_ig_file([
    f"{IG_ROOT}/ads_and_topics/videos_watched.json",
    f"{IG_ROOT}/impressions/videos_watched.json",
    f"{IG_ROOT}/your_topics/videos_watched.json",
    os.path.join(os.path.dirname(IG_ROOT), "ads_information", "ads_and_topics", "videos_watched.json"),
])

def parse_videos_watched(path):
    rows = []
    if not path:
        return rows
    with open(path, encoding="utf-8", errors="replace") as f:
        data = json.load(f)
    items = data if isinstance(data, list) else (data.get("impressions_history_videos_watched") or [])
    for item in items:
        ts = item.get("timestamp", 0)
        if ts:
            rows.append({"author": "", "timestamp": int(ts)})
    return rows

videos_rows = parse_videos_watched(videos_watched_path)
print(f"Vidéos regardées : {len(videos_rows):,}")

if videos_rows:
    df_videos_watched = spark.createDataFrame(videos_rows, schema=schema_pv) \
        .withColumn("event_date",    F.to_timestamp(F.col("timestamp"))) \
        .withColumn("event_year",    F.year("event_date")) \
        .withColumn("event_month",   F.date_format("event_date", "yyyy-MM")) \
        .withColumn("event_hour",    F.hour("event_date")) \
        .withColumn("event_weekday", F.dayofweek("event_date")) \
        .withColumn("platform",      F.lit("instagram")) \
        .withColumn("action_type",   F.lit("video_watched")) \
        .withColumn("interaction_weight", F.lit(0.7))

    df_videos_watched.write.format("delta").mode("overwrite").save(os.path.join(WAREHOUSE, "instagram_videos_watched"))
    print(f"instagram_videos_watched -- {df_videos_watched.count():,} lignes")
else:
    print("⚠ videos_watched.json introuvable ou vide — fichier skippé")

# ── PARTIE 7 — Story Likes ────────────────────────────────────────────────────
story_likes_path = _find_ig_file([
    f"{IG_ROOT}/story_activities/story_likes.json",
    f"{IG_ROOT}/story_interactions/story_likes.json",
    f"{IG_ROOT}/likes/story_likes.json",
    f"{IG_ROOT}/story_likes.json",
])

def parse_story_likes(path):
    rows = []
    if not path:
        return rows
    with open(path, encoding="utf-8", errors="replace") as f:
        data = json.load(f)
    items = data if isinstance(data, list) else (data.get("story_activities_story_likes") or data.get("story_likes") or [])
    for item in items:
        ts = item.get("timestamp", 0)
        if ts:
            rows.append({"author": "", "timestamp": int(ts)})
    return rows

story_rows = parse_story_likes(story_likes_path)
print(f"Story likes : {len(story_rows):,}")

if story_rows:
    df_story_likes = spark.createDataFrame(story_rows, schema=schema_pv) \
        .withColumn("event_date",    F.to_timestamp(F.col("timestamp"))) \
        .withColumn("event_year",    F.year("event_date")) \
        .withColumn("event_month",   F.date_format("event_date", "yyyy-MM")) \
        .withColumn("event_hour",    F.hour("event_date")) \
        .withColumn("event_weekday", F.dayofweek("event_date")) \
        .withColumn("platform",      F.lit("instagram")) \
        .withColumn("action_type",   F.lit("story_like")) \
        .withColumn("interaction_weight", F.lit(1.5))

    df_story_likes.write.format("delta").mode("overwrite").save(os.path.join(WAREHOUSE, "instagram_story_likes"))
    print(f"instagram_story_likes -- {df_story_likes.count():,} lignes")
else:
    print("⚠ story_likes.json introuvable ou vide — fichier skippé")

# ── PARTIE 8 — Recherches Instagram ──────────────────────────────────────────
searches_path = _find_ig_file([
    f"{IG_ROOT}/searches/word_or_phrase_searches.json",
    f"{IG_ROOT}/word_or_phrase_searches.json",
    os.path.join(os.path.dirname(IG_ROOT), "logged_information", "recent_searches", "word_or_phrase_searches.json"),
    os.path.join(os.path.dirname(IG_ROOT), "recent_searches", "word_or_phrase_searches.json"),
])

def parse_ig_searches(path):
    rows = []
    if not path:
        return rows
    with open(path, encoding="utf-8", errors="replace") as f:
        data = json.load(f)
    items = data.get("searches_keyword") or data.get("keyword_searches") or (data if isinstance(data, list) else [])
    for item in items:
        smd   = item.get("string_map_data", {})
        query = (smd.get("Recherche", {}) or smd.get("Search", {})).get("value", "") or item.get("title", "")
        ts    = (smd.get("Heure", {}) or smd.get("Time", {})).get("timestamp", 0)
        if ts and query:
            rows.append({"query": query, "timestamp": int(ts)})
    return rows

search_rows = parse_ig_searches(searches_path)
print(f"Recherches Instagram : {len(search_rows):,}")

schema_search = StructType([
    StructField("query",     StringType(), True),
    StructField("timestamp", LongType(),   True),
])

if search_rows:
    df_ig_searches = spark.createDataFrame(search_rows, schema=schema_search) \
        .withColumn("event_date",    F.to_timestamp(F.col("timestamp"))) \
        .withColumn("event_year",    F.year("event_date")) \
        .withColumn("event_month",   F.date_format("event_date", "yyyy-MM")) \
        .withColumn("event_hour",    F.hour("event_date")) \
        .withColumn("event_weekday", F.dayofweek("event_date")) \
        .withColumn("platform",      F.lit("instagram")) \
        .withColumn("action_type",   F.lit("search")) \
        .withColumn("interaction_weight", F.lit(1.0)) \
        .withColumn("char_count",    F.length("query")) \
        .withColumn("word_count",    F.size(F.split(F.trim("query"), r"\s+")))

    df_ig_searches.write.format("delta").mode("overwrite").save(os.path.join(WAREHOUSE, "instagram_searches"))
    print(f"instagram_searches -- {df_ig_searches.count():,} lignes")
else:
    print("⚠ word_or_phrase_searches.json introuvable ou vide — fichier skippé")

spark.stop()
print("\n✓ Notebook Instagram terminé.")


Posts vus : 820
instagram_posts_viewed -- 820 lignes
Vidéos regardées : 994
instagram_videos_watched -- 994 lignes
Story likes : 366
instagram_story_likes -- 366 lignes
Recherches Instagram : 1
instagram_searches -- 1 lignes

✓ Notebook Instagram terminé.
